# Repeated-structure residual metric ablation

Actual healthy PT cuts are partial observations of repeated ordered structures. This special case integrates their variation through one global training-residual covariance. The canonical scFates axis, mean path and gene decoder stay fixed. It tests a metric, not physical trajectories or probability flow.

In [ ]:
from pathlib import Path
import hashlib, io, json, sys
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
REPO=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pseudospace').is_dir())
sys.path.insert(0,str(REPO))
from pseudospace.repeated_inputs import load_repeated_mouse_inputs
from pseudospace.count_representation import fit_count_representation,transform_count_representation
from pseudospace.repeated_structure import fit_atlas
from pseudospace.trajectory_benchmark import fit_scfates_axis,project_neighbor_axis
from pseudospace.canonical_mean_atlas import fit_mean_atlas,project_mean_atlas
from pseudospace.residual_metric_atlas import fit_residual_metric_atlas,project_residual_metric_atlas
from pseudospace.stage_cache import cached_payload
from importlib.metadata import version
import argparse,os
parser=argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root',default=os.environ.get('PSEUDOSPACE_DATA_ROOT',str(REPO/'data')))
parser.add_argument('--results-root',default=os.environ.get('PSEUDOSPACE_RESULTS_ROOT',str(REPO/'results')))
args,_=parser.parse_known_args()
DATA_ROOT,RESULTS_ROOT=Path(args.data_root).expanduser(),Path(args.results_root).expanduser()
OUT=RESULTS_ROOT/'repeated_structure_residual_metric'
OUT.mkdir(parents=True,exist_ok=True)
REFERENCE_OUT=RESULTS_ROOT/'healthy_reconstruction_trajectory_benchmark'
NOTEBOOK_LOGIC_VERSION='23.residual_metric.2'
SEED,N_COMPONENTS,N_FOLDS=15,15,3
CONTROLS=('Ctrl1A2','Ctrl1A4')
GRID=np.linspace(0,1,101)
SCFATES_PARAMS=dict(nodes=30,seed=SEED)
COUNT_ATLAS_PARAMS=dict(grid=GRID,bandwidth=.08,theta=100.,min_effective=12.)
MARKERS=set('Slc5a2 Slc5a12 Gatm Slc22a6 Slc13a3 Slc1a1 Cyp2e1 Slc7a13 Slc22a7 Slc5a1 Cyp7b1 Slc7a12 Lrp2 Cubn Slc34a1 Slc6a18 Acsm3 Agt'.casefold().split())
VERSIONS={name:version(name) for name in ['scanpy','scFates','anndata','scikit-learn']}
CODE_FILES=[REPO/'pseudospace'/f for f in ['repeated_inputs.py','count_representation.py','trajectory_benchmark.py','repeated_structure.py','canonical_mean_atlas.py','residual_metric_atlas.py','stage_cache.py']]
CODE_DIGEST=hashlib.sha256(b''.join(p.read_bytes() for p in CODE_FILES)).hexdigest()
BASELINE_SCORE_PATH=REFERENCE_OUT/'scores.csv'
if not BASELINE_SCORE_PATH.is_file(): raise FileNotFoundError(f'Required notebook-21 baseline is missing: {BASELINE_SCORE_PATH}')
BASELINE_SCORE_DIGEST=hashlib.sha256(BASELINE_SCORE_PATH.read_bytes()).hexdigest()
CACHE_PARAMS=dict(version=NOTEBOOK_LOGIC_VERSION,controls=CONTROLS,seed=SEED,
    components=N_COMPONENTS,folds=N_FOLDS,markers=sorted(MARKERS),grid=GRID.tolist(),
    scfates=SCFATES_PARAMS,mean_atlas=dict(bandwidth=.08,min_effective=12.),
    count_atlas={**COUNT_ATLAS_PARAMS,'grid':GRID.tolist()},
    count_representation_theta=100.,projection_neighbors=15,gene_spline_ridge=.001,
    quarter_read_fraction=.25,quarter_read_seed_rule='seed+fold',
    baseline_score_digest=BASELINE_SCORE_DIGEST,versions=VERSIONS)
print('Installed benchmark versions:',VERSIONS)

PREVIOUS_OUT=RESULTS_ROOT/'scfates_repeated_count_reference'
PREVIOUS_NAMES=['scores','positions','thin_scores','stability','fixed_program_scores']
for name in PREVIOUS_NAMES:
    if not (PREVIOUS_OUT/(name+'.csv')).is_file(): raise FileNotFoundError('Run notebook 22 first: '+name)
previous={name:pd.read_csv(PREVIOUS_OUT/(name+'.csv')) for name in PREVIOUS_NAMES}
PANEL_PATH=RESULTS_ROOT/'within_segment_repeated_structure_validation'/'selected_genes.csv'
frozen_panel=pd.read_csv(PANEL_PATH,usecols=['source_specimen','test_specimen','fold','segment','gene','training_cv_gain'])
if frozen_panel.duplicated(['source_specimen','fold','segment','gene']).any(): raise ValueError('Duplicate frozen panel gene')
CACHE_PARAMS['residual_covariance']=dict(estimator='LedoitWolf',assume_centered=True,eigen_floor_relative=1e-6,eigen_floor_absolute=1e-8,residual_positions='original scFates training z')

CACHE_PARAMS['matched_spherical_control']='trace(fitted covariance)/nfeatures * identity'


## Validated measured inputs

Same structure filter, markers and gene folds as notebooks 21–22. Depth is unused.

Individual marker and response-fold genes are excluded from modeling features. Total-library exposure includes all measured genes, so aggregate compositional dependence remains. These tests are not fully independent of the withheld measurements.

In [ ]:
healthy=load_repeated_mouse_inputs(DATA_ROOT,samples=CONTROLS)
obs=healthy.obs.copy()
ids=obs.structure_id.to_numpy(str)
specimens=obs['sample'].astype(str).to_numpy()
labels=obs.segment_class.astype(str).str.replace('PT-','',regex=False).to_numpy()
if set(np.unique(specimens))!=set(CONTROLS): raise ValueError('Expected exactly the two healthy control specimens')
if not np.isin(labels,['S1','S2','S3']).all(): raise ValueError('Resolve uncertain PT labels upstream')
anatomy=np.array([{'S1':0,'S2':1,'S3':2}[x] for x in labels],dtype=int)
genes=healthy.var_names.to_numpy(str)
X=healthy.layers['lognorm']; X=X.toarray().astype(np.float64,copy=False) if sparse.issparse(X) else np.asarray(X,dtype=np.float64)
raw_counts=healthy.layers['counts']
raw_total=np.asarray(raw_counts.sum(axis=1)).ravel()
if not np.isfinite(X).all() or not np.isfinite(raw_total).all() or np.any(raw_total<=0): raise ValueError('Invalid expression or all-gene exposure')
if 'n_spots' not in obs: raise ValueError('Required n_spots metadata is missing')
n_spots=pd.to_numeric(obs.n_spots,errors='coerce').to_numpy(float)
if not np.isfinite(n_spots).all() or np.any(n_spots<0): raise ValueError('Invalid n_spots metadata')
marker_mask=np.array([g.casefold() in MARKERS for g in genes])
gene_fold=np.array([int(hashlib.sha256(g.casefold().encode()).hexdigest()[:8],16)%N_FOLDS for g in genes])
manifest=dict(healthy.uns['repeated_mouse_input_manifest'])
manifest.update(notebook_logic_version=NOTEBOOK_LOGIC_VERSION,config=CACHE_PARAMS,code_digest=CODE_DIGEST,markers_excluded=sorted(MARKERS),dpt_used_for_fitting=False,depth_used=False)
(OUT/'input_manifest.json').write_text(json.dumps(manifest,indent=2))
display(pd.crosstab(specimens,labels))
print('Structures:',len(ids),'genes:',len(genes),'outer split: whole specimen')

## Fixed protocol

Source-only Ledoit–Wolf covariance, no query fitting or parameter sweep. Frozen notebook-18 programs retain their earlier atlas selection bias. Query anatomy enters scoring only. Quarter-read predictions use the original response measurements as fixed targets.

After the first covariance run, a matched spherical residual-scale control was added to isolate geometry. That diagnostic is recorded as post-result in the research ledger. Both arms share the training covariance trace; neither uses query tuning.

In [ ]:
def dense_panel(matrix,rows,columns):
    value=matrix[rows][:,columns]
    return value.toarray() if sparse.issparse(value) else np.asarray(value)

def fit_gene_spline(response,z,sample):
    return fit_atlas(response,z,sample,grid=GRID,ridge=.001)

def normalized_variance(response,sample):
    mean=np.mean([response[sample==s].mean(axis=0) for s in np.unique(sample)],axis=0)
    return np.maximum(np.mean([((response[sample==s]-mean)**2).mean(axis=0) for s in np.unique(sample)],axis=0),1e-3)

def primary_rows(response,predictions,variance,query_anatomy,meta):
    out=[]
    for method,pred in predictions.items():
        scaled=(response-pred)**2/variance
        for seg,name in [(None,'all'),(0,'S1'),(1,'S2'),(2,'S3')]:
            use=np.ones(len(response),bool) if seg is None else query_anatomy==seg
            if use.any(): out.append(dict(**meta,method=method,segment=name,n=int(use.sum()),normalized_mse=float(scaled[use].mean())))
    return out


In [ ]:
def run_metric_ablation():
    score_rows,thin_rows,stability_rows,position_rows,selected_rows,diagnostics,failures=[],[],[],[],[],[],[]
    from scipy.stats import spearmanr
    for source in CONTROLS:
        target=next(s for s in CONTROLS if s!=source)
        tr,te=specimens==source,specimens==target
        for fold in range(N_FOLDS):
            key=dict(source_specimen=source,test_specimen=target,fold=fold)
            try:
                mg=np.flatnonzero(~marker_mask&(gene_fold!=fold));eg=np.flatnonzero(~marker_mask&(gene_fold==fold))
                train_counts=dense_panel(raw_counts,tr,mg);query_counts=dense_panel(raw_counts,te,mg)
                Y,transform=fit_count_representation(train_counts,raw_total[tr],specimens[tr],N_COMPONENTS,theta=100.,seed=SEED)
                query_Y=transform_count_representation(query_counts,raw_total[te],transform)
                train_z,meta=fit_scfates_axis(Y,anatomy[tr],**SCFATES_PARAMS)
                atlas=fit_residual_metric_atlas(Y,train_z,grid=GRID,bandwidth=.08,min_effective=12)
                cov_query=project_residual_metric_atlas(atlas,query_Y)
                residual_scale=float(np.trace(atlas['covariance'])/Y.shape[1])
                spherical_atlas={**atlas,'covariance':np.eye(Y.shape[1])*residual_scale,'precision':np.eye(Y.shape[1])/residual_scale,'logdet':Y.shape[1]*np.log(residual_scale)}
                spherical_query=project_residual_metric_atlas(spherical_atlas,query_Y)
                iso_query=project_mean_atlas(atlas,query_Y)
                nn_query=project_neighbor_axis(Y,train_z,query_Y,k=15)
                coords={'residual_covariance':cov_query['z_mean'],'residual_spherical':spherical_query['z_mean'],'mean_reference':iso_query['z_mean'],'scFates_neighbor':nn_query}
                for method in ['mean_reference','scFates_neighbor']:
                    saved=previous['positions'].query('source_specimen==@source and fold==@fold and method==@method').set_index('structure_id')
                    if saved.index.duplicated().any() or set(saved.index)!=set(ids[te]): raise ValueError('Reference profile identity mismatch')
                    if not np.allclose(coords[method],saved.loc[ids[te],'z'],rtol=1e-8,atol=1e-10): raise ValueError('Frozen coordinate reproduction failed: '+method)
                train_response=X[tr][:,eg];query_response=X[te][:,eg]
                variance=normalized_variance(train_response,specimens[tr])
                decoder=fit_gene_spline(train_response,train_z,specimens[tr])
                predictions={method:decoder.predict(z) for method,z in coords.items()}
                scores=pd.DataFrame(primary_rows(query_response,predictions,variance,anatomy[te],key))
                old=previous['scores'].query('source_specimen==@source and fold==@fold and method!="count_NB_reference"')
                joined=scores[scores.method.isin(['mean_reference','scFates_neighbor'])].merge(old,on=['method','segment'],suffixes=('_new','_old'),validate='one_to_one')
                if len(joined)!=8 or not np.allclose(joined.normalized_mse_new,joined.normalized_mse_old,rtol=1e-8,atol=1e-10): raise ValueError('Reference all-gene scores did not reproduce')
                score_rows.extend(scores.to_dict('records'))
                gene_to_j={g:j for j,g in enumerate(genes[eg])}
                panel=frozen_panel.query('source_specimen==@source and test_specimen==@target and fold==@fold')
                for seg,segname in enumerate(['S1','S2','S3']):
                    selected=panel[panel.segment==segname]
                    if selected.empty or not set(selected.gene).issubset(gene_to_j): raise ValueError('Missing or invalid frozen program: '+segname)
                    jj=np.array([gene_to_j[g] for g in selected.gene],int)
                    scale=np.maximum(np.var(train_response[anatomy[tr]==seg],axis=0),1e-3)[jj]
                    use=anatomy[te]==seg
                    for method,pred in predictions.items():
                        error=float(np.mean((query_response[use][:,jj]-pred[use][:,jj])**2/scale))
                        selected_rows.append(dict(**key,method=method,segment=segname+'_selected',n=int(use.sum()),selected_gene_count=len(jj),normalized_mse=error))
                        if method in ['mean_reference','scFates_neighbor']:
                            selected_name=segname+'_selected'
                            prior=previous['fixed_program_scores'].query('source_specimen==@source and fold==@fold and method==@method and segment==@selected_name')
                            if len(prior)!=1 or not np.isclose(error,prior.normalized_mse.iloc[0],rtol=1e-8,atol=1e-10): raise ValueError('Frozen-program reference score did not reproduce')
                full=sparse.csr_matrix(raw_counts[te]);rng=np.random.default_rng(SEED+fold)
                thinned=sparse.csr_matrix((rng.binomial(full.data.astype(np.int64),.25),full.indices.copy(),full.indptr.copy()),shape=full.shape)
                library=np.asarray(thinned.sum(axis=1)).ravel().astype(float)
                thin_counts=dense_panel(thinned,np.ones(len(library),bool),mg)
                thin_Y=transform_count_representation(thin_counts,library,transform)
                cov_thin=project_residual_metric_atlas(atlas,thin_Y);iso_thin=project_mean_atlas(atlas,thin_Y)
                spherical_thin=project_residual_metric_atlas(spherical_atlas,thin_Y)
                thin_coords={'residual_covariance':cov_thin['z_mean'],'residual_spherical':spherical_thin['z_mean'],'mean_reference':iso_thin['z_mean'],'scFates_neighbor':project_neighbor_axis(Y,train_z,thin_Y,k=15)}
                thin_pred={method:decoder.predict(z) for method,z in thin_coords.items()}
                thin_rows.extend(primary_rows(query_response,thin_pred,variance,anatomy[te],key))
                for method,z in coords.items():
                    thinz=thin_coords[method];movement=np.abs(thinz-z)
                    entropy=cov_query['entropy'] if method=='residual_covariance' else spherical_query['entropy'] if method=='residual_spherical' else iso_query['entropy'] if method=='mean_reference' else np.full(len(z),np.nan)
                    for seg,name in [(None,'all'),(0,'S1'),(1,'S2'),(2,'S3')]:
                        use=np.ones(len(z),bool) if seg is None else anatomy[te]==seg
                        stability_rows.append(dict(**key,method=method,segment=name,n=int(use.sum()),coordinate_mae=float(movement[use].mean()),fraction_moved_over_0_1=float(np.mean(movement[use]>.1)),within_segment_spearman=float(spearmanr(z[use],thinz[use]).statistic),conditional_entropy_mean=float(entropy[use].mean())))
                    for structure_id,value,h,seg in zip(ids[te],z,entropy,anatomy[te]): position_rows.append(dict(**key,method=method,structure_id=structure_id,z=float(value),conditional_entropy=float(h),segment=['S1','S2','S3'][int(seg)]))
                diagnostics.append(dict(**key,status='complete',shrinkage=float(atlas['covariance_shrinkage']),covariance_condition=float(np.linalg.cond(atlas['covariance'])),residual_scale=residual_scale,nearest_curve_scale=float(atlas['sigma2']),metadata=json.dumps(meta)))
                print('Residual covariance:',source,'→',target,'fold',fold,flush=True)
            except Exception as exc:
                failures.append(dict(**key,error=str(exc)));diagnostics.append(dict(**key,status='failed',shrinkage=np.nan,covariance_condition=np.nan,metadata=''))
    frames=dict(scores=score_rows,thin_scores=thin_rows,selected_scores=selected_rows,stability=stability_rows,positions=position_rows,diagnostics=diagnostics,failures=failures)
    return {name:pd.DataFrame(rows).to_json(orient='table') for name,rows in frames.items()}


In [ ]:
CACHE_INPUTS=dict(X=X,counts=raw_counts,library=raw_total,ids=ids,specimens=specimens,labels=labels,n_spots=n_spots,gene_fold=gene_fold,markers=marker_mask,frozen_panel=frozen_panel,previous=previous)
payload=cached_payload('repeated_structure_residual_metric',run_metric_ablation,root=OUT/'stage_cache',params=CACHE_PARAMS,inputs=CACHE_INPUTS,code=CODE_DIGEST)
outputs={name:pd.read_json(io.StringIO(str(value)),orient='table') for name,value in payload.items()}
for name,frame in outputs.items():frame.to_csv(OUT/(name+'.csv'),index=False)
if len(outputs['diagnostics'])!=6 or len(outputs['failures']): raise RuntimeError('Incomplete residual covariance experiment; inspect failures.csv')
display(outputs['diagnostics'])
for name in ['scores','selected_scores']:
    frame=outputs[name]
    extra=previous['scores'].query('method=="count_NB_reference"') if name=='scores' else previous['fixed_program_scores'].query('method in ["count_NB_reference","quadratic_segment_coverage_oracle"]')
    frame=pd.concat([frame,extra],ignore_index=True)
    base=frame.query('method=="scFates_neighbor"')[['source_specimen','fold','segment','normalized_mse']].rename(columns={'normalized_mse':'baseline_mse'})
    comparison=frame.merge(base,on=['source_specimen','fold','segment'])
    comparison['gain_vs_scfates_percent']=100*(1-comparison.normalized_mse/comparison.baseline_mse)
    if name=='selected_scores':
        oracle=frame.query('method=="quadratic_segment_coverage_oracle"')[['source_specimen','fold','segment','normalized_mse']].rename(columns={'normalized_mse':'quadratic_mse'})
        comparison=comparison.merge(oracle,on=['source_specimen','fold','segment'],validate='many_to_one')
        comparison['gain_vs_quadratic_percent']=100*(1-comparison.normalized_mse/comparison.quadratic_mse)
    comparison.to_csv(OUT/(name+'_comparison.csv'),index=False)
    display(comparison.groupby(['source_specimen','segment','method'])[['normalized_mse','gain_vs_scfates_percent']].mean())
display(outputs['stability'].groupby(['source_specimen','segment','method'])[['coordinate_mae','fraction_moved_over_0_1','within_segment_spearman']].mean())

# The matched control isolates covariance geometry from residual scale.
metric_pair=outputs['scores'].query('method in ["residual_covariance","residual_spherical"]').pivot(index=['source_specimen','fold','segment'],columns='method',values='normalized_mse')
metric_pair['covariance_gain_vs_matched_spherical_percent']=100*(1-metric_pair.residual_covariance/metric_pair.residual_spherical)
metric_pair.to_csv(OUT/'matched_metric_comparison.csv')
display(metric_pair.groupby(['source_specimen','segment']).mean())


## Measured comparisons

Numerical axis and conditional posteriors are not anatomical precision. Gene folds and reused healthy mice are descriptive development evidence.

In [ ]:
import matplotlib.pyplot as plt
selected=pd.read_csv(OUT/'selected_scores_comparison.csv')
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,source in zip(axes,CONTROLS):
    for method,offset,color,label in [('residual_covariance',-.08,'C3','residual covariance'),('residual_spherical',.08,'C1','matched spherical')]:
        for i,seg in enumerate(['S1_selected','S2_selected','S3_selected']):
            values=selected.query('source_specimen==@source and segment==@seg and method==@method').gain_vs_scfates_percent.to_numpy()
            ax.scatter(i+offset+np.linspace(-.03,.03,len(values)),values,color=color,label=label if i==0 else None)
            ax.plot([i+offset-.06,i+offset+.06],[values.mean()]*2,color=color)
    ax.axhline(0,color='gray',linewidth=.7)
    ax.set(xticks=[0,1,2],xticklabels=['S1','S2','S3'],title=source+' → held-out control',ylabel='Program error reduction vs neighbors (%)')
axes[0].legend(frameon=False,fontsize=9)
fig.suptitle('Global training-residual metric; frozen gene panels')
fig.tight_layout();fig.savefig(OUT/'residual_metric_program_prediction.png',dpi=160);plt.show()


## Interpretation of the measured result

The covariance reference does not improve held-out prediction or read stability over neighbor projection. Covariance geometry is worse than its matched spherical control with Ctrl1A2 as source and identical with Ctrl1A4, whose covariance shrinks fully to a sphere. The reverse S2 positional-program weakness remains. Neither residual covariance nor a probability-flow field is adopted. The next hypothesis is restricted specimen registration, specified in the research ledger; healthy calibration, injury biology and spatial position must remain distinct.
